# GEQIE QML precompute example

## Imports

In [1]:
import geqie_qml
import torch
import torchvision

import numpy as np

## Dataset fetch

In [2]:
mnist_train = torchvision.datasets.MNIST(root="./.data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST(root="./.data", train=False, download=True)

print(f"Train: {len(mnist_train)}, Test: {len(mnist_test)}")

Train: 60000, Test: 10000


## Image resizing

Using `torch.nn.functional.interpolate` to eagerly resize the dataset images.

In [ ]:
TARGET_SIZE = 8

def resize_dataset(data: torch.Tensor, size: int = TARGET_SIZE) -> np.ndarray:
    # interpolate expects (N, C, H, W) floats
    # torch hack to eagerly resize the entire dataset at once
    resized = torch.nn.functional.interpolate(
        data.unsqueeze(1).float(),
        size=(size, size),
        mode="bilinear",
        align_corners=False,
    ).squeeze(1)  # back to (N, H, W)
    return resized.numpy().astype(np.uint8)

train_data = resize_dataset(mnist_train.data)
test_data  = resize_dataset(mnist_test.data)

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape:  {test_data.shape}")

Train data shape: (60000, 8, 8)
Test data shape:  (10000, 8, 8)


In [4]:
train_data, train_labels = train_data, mnist_train.targets
test_data, test_labels = test_data, mnist_test.targets

## Precompute

In [5]:
geqie_qml.compute_and_save_circuits(
    data=train_data,
    labels=train_labels,
    save_dir=".precomputed/train",
    geqie_encoding="frqi",
    encoding_params={},
    number_of_workers=None,
)

Processing tasks: 100%|██████████| 60000/60000 [51:09<00:00, 19.55it/s]  


In [6]:
geqie_qml.compute_and_save_circuits(
    data=test_data,
    labels=test_labels,
    save_dir=".precomputed/test",
    geqie_encoding="frqi",
    encoding_params={},
    number_of_workers=None,
)

Processing tasks: 100%|██████████| 10000/10000 [08:49<00:00, 18.89it/s]
